Importing all the necessary libraries

In [ ]:
import pandas as pd
import numpy as np

# Graphs and plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

# uciml repo rather than local dataset
from ucimlrepo import fetch_ucirepo

# Suppressing warnings for cleaner output
# import warnings
# warnings.filterwarnings('ignore')

print("All required libraries are loaded successfully!")

: 

Fetching the dataset and conversions

In [ ]:
# Fetch the dataset from uciml repo
banknote_data = fetch_ucirepo(id=267)

# df to use the data
banknote_df = pd.DataFrame(data=banknote_data.data.features, columns=banknote_data.data.features.columns)

# Add the class variable
banknote_df['authentic'] = banknote_data.data.targets

print(f"Rows {banknote_df.shape[0]} columns {banknote_df.shape[1]}")

features = banknote_df.drop('authentic', axis=1)
target = banknote_df['authentic']

print("\nTarget column value distribution:")
print(banknote_df['authentic'].value_counts())

banknote_df.head()

In [ ]:
print("Our models with pipelines")

# Pipeline for Logistic Regression
log_pl = Pipeline([
    ('feature_scaling', StandardScaler()),  
    ('logistic_classifier', LogisticRegression(max_iter=1000, random_state=67))
])
# the max iters ensure that the model converges by those iterations
# if not the model reports us the error

# Pipeline for Random Forest
rf_pl = Pipeline([
    ('feature_scaling', StandardScaler()),  
    ('forest_classifier', RandomForestClassifier(n_estimators=100, random_state=67))
])
# the n_esimators is the number of trees in the forest
# the random state is used to ensure there is randomness in the model i.e different results on different runs 
# non-deterministic diverse model creation

models = {
    "Logistic Regression": log_pl,
    "Random Forest": rf_pl
}

for model_name in models:
    print(f" - {model_name}")

In [ ]:
# Function to evaluate model performance
def eval_model_perf(trained_model, test_features, test_labels):
    
    # Get predictions from the model
    predictions = trained_model.predict(test_features)
    
    # Calculate different performance metrics
    performance_metrics = {
        "Accuracy": accuracy_score(test_labels, predictions),
        "Precision": precision_score(test_labels, predictions),
        "Recall": recall_score(test_labels, predictions),
        "F1_Score": f1_score(test_labels, predictions)
    }
    
    return performance_metrics

print("Model evaluation function is ready to use!")

In [ ]:
print("Different Splitting Approaches")
print("-" * 50)

# Three different splits
split_configurations = {
    '0.8 train': 0.2,
    '0.7 train': 0.3, 
    '0.6 train': 0.4
}

# Store all results 
all_holdout_results = {}

for split_name, test_size in split_configurations.items():
    print(f"\n{'-'*30}")
    print(f"Testing: {split_name}")
    print(f"{'-'*30}")
    
    # Split our data
    x_train, x_test, y_train, y_test = train_test_split(
        features, target, 
        test_size=test_size, 
        random_state=67, 
        stratify=target  
    )
    
    print(f"Training samples: {x_train.shape[0]}")
    print(f"Testing samples: {x_test.shape[0]}")
    print(f"Class distribution in test set:\n{y_test.value_counts()}")
    
    current_split_results = {}
    
    # Test each model on this split
    for model_name, ml_model in models.items():
        print(f"\n--- Evaluating {model_name} ---")
        
        # Train the model
        ml_model.fit(x_train, y_train)
        
        # Get predictions and evaluate
        test_predictions = ml_model.predict(x_test)
        current_split_results[model_name] = eval_model_perf(ml_model, x_test, y_test)
        
        # Print the performance metrics values
        for metric_name, metric_value in current_split_results[model_name].items():
            print(f"  {metric_name}: {metric_value:.3f}")
    
    # Save results for this split configuration
    all_holdout_results[split_name] = current_split_results

print("\nFinished testing all split configurations!")

In [ ]:
# Using cross-validation for more robust evaluation
print("Starting Cross-Validation Analysis")
print("-" * 45)

# Set up 10-fold cross-validation
# using stratified k-fold to maintain class distribution in each fold
# means that in each fold the number of positive and negative samples is
# approximately the same as in the entire dataset this ensures some kind of uniformity
cross_val_setup = StratifiedKFold(n_splits=10, shuffle=True, random_state=67)
cross_val_results = {}

for model_name, ml_model in models.items():
    print(f"\nCross-validating {model_name}:")
    
    # Performance metrics
    evaluation_metrics = {
        'accuracy': 'accuracy',
        'precision': 'precision',
        'recall': 'recall', 
        'f1': 'f1'
    }
    
    model_cv_scores = {}
    
    for metric_name, sklearn_metric in evaluation_metrics.items():
        # Perform cross-validation for this specific metric
        cv_scores = cross_val_score(ml_model, features, target, cv=cross_val_setup, scoring=sklearn_metric)
        
        model_cv_scores[metric_name] = {
            'average_score': np.mean(cv_scores),
            'score_std': np.std(cv_scores),
            'all_scores': cv_scores
        }
        
        print(f"  {metric_name}: {np.mean(cv_scores):.3f} (±{np.std(cv_scores):.3f})")
    
    cross_val_results[model_name] = model_cv_scores

print("\nCross-validation analysis completed!")

In [ ]:
# Detailed comparison of all methods
print("Comprehensive Performance Comparison")
print("-" * 50)

# We'll build a results table from all our experiments
results_table = []

for model_name in models.keys():
    # Add cross-validation results first
    results_table.append({
        'Model': model_name,
        'Validation_Method': '10-Fold Cross-Validation',
        'Accuracy': cross_val_results[model_name]['accuracy']['average_score'],
        'Precision': cross_val_results[model_name]['precision']['average_score'],
        'Recall': cross_val_results[model_name]['recall']['average_score'],
        'F1_Score': cross_val_results[model_name]['f1']['average_score'],
        'Stability_Measure': cross_val_results[model_name]['accuracy']['score_std']
    })
    
    # Add results from each train-test split
    for split_name, split_result in all_holdout_results.items():  
        results_table.append({
            'Model': model_name,
            'Validation_Method': f'Hold-Out {split_name}',
            'Accuracy': split_result[model_name]['Accuracy'],
            'Precision': split_result[model_name]['Precision'],
            'Recall': split_result[model_name]['Recall'],
            'F1_Score': split_result[model_name]['F1_Score'],
            'Stability_Measure': 0.0 # because single split, no stability measure
        })

# Create a clean dataframe for display
comparison_dataframe = pd.DataFrame(results_table)

print("Detailed Performance Results:")
print(comparison_dataframe.to_string(index=False, float_format='%.3f'))

In [ ]:
# Summary of our findings
print("Overall Performance Summary")
print("-" * 40)

for model_name in models.keys():
    print(f"\nPerformance Summary for {model_name}:")
    print("-" * 40)
    
    # Cross-validation performance
    cv_accuracy = cross_val_results[model_name]['accuracy']['average_score']
    cv_std = cross_val_results[model_name]['accuracy']['score_std']
    print(f"Cross-Validation Accuracy: {cv_accuracy:.3f} (±{cv_std:.3f})")
    
    # Hold-out performance across different splits
    for split_name in split_configurations.keys():
        ho_accuracy = all_holdout_results[split_name][model_name]['Accuracy']
        print(f"Hold-Out {split_name} Accuracy: {ho_accuracy:.3f}")
    
    # Stability assessment
    if cv_std < 0.02:
        stability = "Excellent"
    elif cv_std < 0.05:
        stability = "Good"
    else:
        stability = "Variable"
    
    print(f"Model Stability: {stability}")

In [ ]:
# Creating visualizations to compare performance
print("Generating Performance Comparison Charts")
print("-" * 50)

# Prepare data for visualization
viz_data = []

# 3 hold-out splits + 1 cross-validation = 4 total
validation_methods = ['Cross-Validation'] + list(split_configurations.keys())

for model_name in models.keys():
    # Cross-validation data
    viz_data.append({
        'Model': model_name,
        'Method': 'Cross-Validation',
        'Accuracy': cross_val_results[model_name]['accuracy']['average_score'],
        'Error_Margin': cross_val_results[model_name]['accuracy']['score_std']
    })
    
    # Hold-out data for each split
    for split_name in split_configurations.keys():
        viz_data.append({
            'Model': model_name,
            'Method': split_name,
            'Accuracy': all_holdout_results[split_name][model_name]['Accuracy'],
            'Error_Margin': 0.0  # No error margin for single hold-out
        })

viz_df = pd.DataFrame(viz_data)

# Create the main comparison plot
plt.figure(figsize=(14, 7))

# Different colors for each model
model_colors = {'Logistic Regression': 'skyblue', 'Random Forest': 'lightcoral'}

# We'll create positions for each validation method
x_positions = np.arange(len(validation_methods))
bar_width = 0.30

for i, model_name in enumerate(models.keys()):
    model_viz_data = viz_df[viz_df['Model'] == model_name]
    
    # Ensure data is in the correct order
    accuracies = []
    errors = []
    for method in validation_methods:
        method_data = model_viz_data[model_viz_data['Method'] == method]
        if len(method_data) > 0:
            accuracies.append(method_data['Accuracy'].values[0])
            errors.append(method_data['Error_Margin'].values[0])
        else:
            accuracies.append(0)
            errors.append(0)
    
    # Create bars for each method
    bars = plt.bar(x_positions + i * bar_width, accuracies, bar_width,
                   label=model_name, color=model_colors[model_name], 
                   alpha=0.8, edgecolor='black')

plt.xlabel('Validation Method', fontsize=12, fontweight='bold')
plt.ylabel('Accuracy Score', fontsize=12, fontweight='bold')
plt.title('Model Accuracy Across Different Validation Approaches', fontsize=14, fontweight='bold')
plt.xticks(x_positions + bar_width/2, validation_methods, rotation=0)
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0.85, 1.02)

# # Add some explanatory text
# plt.figtext(0.5, 0.01, 
#            'Note: Error bars show standard deviation for Cross-Validation only', 
#            ha='center', fontsize=10, style='italic')

plt.tight_layout()
plt.subplots_adjust(bottom=0.1)  # Make space for the footnote
plt.show()

print("\nInsights:")
print(f"• Cross-Validation provides the most reliable estimate with error margins")
print(f"• Hold-Out splits show performance on specific data partitions")
print(f"• Consistent performance across methods indicates model robustness")

In [ ]:
# Creating a detailed comparison of all performance metrics
print("Detailed Performance Metrics Comparison")
print("-" * 55)

# Let's create a comprehensive view of all metrics across different validation methods
metrics_to_compare = ['Accuracy', 'Precision', 'Recall', 'F1_Score']
validation_methods = ['Cross-Validation'] + list(split_configurations.keys())

# Set up our plotting area
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

# Colors for our models
model_colors = {'Logistic Regression': '#3498db', 'Random Forest': '#e74c3c'}

for idx, metric in enumerate(metrics_to_compare):
    # Prepare data specifically for this metric
    metric_comparison_data = []
    
    for model_name in models.keys():
        # Get cross-validation results for this metric
        if metric == 'Accuracy':
            cv_value = cross_val_results[model_name]['accuracy']['average_score']
        else:
            # Map metric names to CV dictionary keys
            cv_metric_key = metric.lower() if metric != 'F1_Score' else 'f1'
            cv_value = cross_val_results[model_name][cv_metric_key]['average_score']
        
        metric_comparison_data.append({
            'Model': model_name,
            'Method': 'CV',
            'Value': cv_value
        })
        
        # Hold-out results for each split
        for split_name in split_configurations.keys():
            # Adjust metric name for hold-out dictionary
            ho_metric_name = metric if metric != 'F1_Score' else 'F1_Score'
            ho_value = all_holdout_results[split_name][model_name][ho_metric_name]
            
            metric_comparison_data.append({
                'Model': model_name,
                'Method': split_name,
                'Value': ho_value
            })
    
    # Convert to DataFrame for easier plotting
    metric_df = pd.DataFrame(metric_comparison_data)
    
    # Create positions for our bars
    method_count = len(validation_methods)
    bar_positions = np.arange(method_count)
    bar_width = 0.35
    
    # Plot each model's results
    for i, model_name in enumerate(models.keys()):
        model_data = metric_df[metric_df['Model'] == model_name]
        values = [model_data[model_data['Method'] == method]['Value'].values[0] 
                 for method in ['CV'] + list(split_configurations.keys())]
        
        # Create bars with slight offset for each model
        offset = i * bar_width
        bars = axes[idx].bar(bar_positions + offset, values, bar_width, 
                           label=model_name, color=model_colors[model_name],
                           alpha=0.8, edgecolor='black')
        
        # Add value labels on top of bars
        for j, value in enumerate(values):
            axes[idx].text(bar_positions[j] + offset, value + 0.01, 
                         f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Customize each subplot
    axes[idx].set_title(f'{metric.replace("_", " ")} Comparison', 
                       fontsize=14, fontweight='bold', pad=15)
    axes[idx].set_xlabel('Validation Method', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel(metric.replace('_', ' '), fontsize=11, fontweight='bold')
    axes[idx].set_xticks(bar_positions + bar_width/2)
    axes[idx].set_xticklabels(['CV'] + list(split_configurations.keys()))
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)
    axes[idx].set_ylim(0.85, 1.02)

plt.tight_layout()
plt.suptitle('Comprehensive Performance Metrics Across All Validation Methods', 
             fontsize=16, fontweight='bold', y=1.02)
plt.show()

print("Detailed metrics comparison completed!")

In [ ]:
# Confusion matrices
print("Confusion Matrices for Model Performance Analysis")
print("-" * 60)

# Let's use the 80/20 split for our confusion matrix analysis since it's most common
selected_split = '0.8 train'

# We need to recreate this split to get the actual test data
X_train_cm, X_test_cm, y_train_cm, y_test_cm = train_test_split(
    features, target, test_size=0.2, random_state=67, stratify=target
)

print(f"Using {selected_split} for confusion matrix analysis")
print(f"Test set size: {X_test_cm.shape[0]} samples")

# Create a figure for our confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Train models and generate predictions for confusion matrices
for idx, (model_name, ml_model) in enumerate(models.items()):
    print(f"\nTraining {model_name} for confusion matrix...")
    
    # Train the model
    ml_model.fit(X_train_cm, y_train_cm)
    
    # Generate predictions
    predictions = ml_model.predict(X_test_cm)
    
    # Calculate confusion matrix
    cm = confusion_matrix(y_test_cm, predictions)
    
    # Create a beautiful confusion matrix plot
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                cbar_kws={'shrink': 0.8}, annot_kws={'size': 14, 'weight': 'bold'})
    
    # Customize labels and titles
    axes[idx].set_title(f'Confusion Matrix - {model_name}\n({selected_split})', 
                       fontsize=14, fontweight='bold', pad=15)
    axes[idx].set_xlabel('Predicted Label', fontsize=9, fontweight='bold')
    axes[idx].set_ylabel('Actual Label', fontsize=9, fontweight='bold')
    axes[idx].set_xticklabels(['Fake (0)', 'Genuine (1)'], fontsize=11)
    axes[idx].set_yticklabels(['Fake (0)', 'Genuine (1)'], fontsize=11, rotation=0)

plt.tight_layout()
plt.suptitle('Confusion Matrix Analysis: Model Performance Comparison', 
             fontsize=16, fontweight='bold', y=1.02)
plt.show()

for model_name, ml_model in models.items():
    # Re-train and predict to get fresh results
    ml_model.fit(X_train_cm, y_train_cm)
    predictions = ml_model.predict(X_test_cm)
    cm = confusion_matrix(y_test_cm, predictions)
    
    # Extract values from confusion matrix
    tn, fp, fn, tp = cm.ravel()
    
    # Calculate additional metrics
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    # print(f"\n {model_name} Detailed Performance:")
    # print(f"   True Positives (TP): {tp} - Correctly identified genuine banknotes")
    # print(f"   True Negatives (TN): {tn} - Correctly identified fake banknotes") 
    # print(f"   False Positives (FP): {fp} - Fake banknotes misclassified as genuine")
    # print(f"   False Negatives (FN): {fn} - Genuine banknotes misclassified as fake")
    # print(f"   Accuracy: {accuracy:.3f}")
    # print(f"   Precision: {precision:.3f} (When predicting genuine, how often correct)")
    # print(f"   Recall: {recall:.3f} (How many genuine banknotes were correctly identified)")
    # print(f"   Specificity: {specificity:.3f} (How many fake banknotes were correctly identified)")
    # print(f"   F1-Score: {f1_score:.3f}")

In [ ]:
# Final comprehensive analysis and actionable recommendations
print("FINAL COMPREHENSIVE ANALYSIS & RECOMMENDATIONS")
print("-" * 60)

print("\n EXECUTIVE SUMMARY")
print("-" * 40)

# Calculate overall best performer
best_performers = {}

# Cross-validation champion
cv_winner = max(models.keys(), 
                key=lambda x: cross_val_results[x]['accuracy']['average_score'])
best_performers['Cross-Validation'] = cv_winner

# Hold-out champions for each split 
for split_name in split_configurations.keys():
    ho_winner = max(models.keys(), 
                   key=lambda x: all_holdout_results[split_name][x]['Accuracy'])
    best_performers[f'Hold-Out {split_name}'] = ho_winner

print("Top models by validation:")
for method, winner in best_performers.items():
    if method == 'Cross-Validation':
        score = cross_val_results[winner]['accuracy']['average_score']
        print(f"   {method}: {winner} (Accuracy: {score:.3f})")
    else:
        # Extract the correct split name from the method string
        # Method looks like: 'Hold-Out 80% Training / 20% Testing'
        # We need just '80% Training / 20% Testing'
        split_full_name = method.replace('Hold-Out ', '')
        score = all_holdout_results[split_full_name][winner]['Accuracy']  
        print(f"   {method}: {winner} (Accuracy: {score:.3f})")

# Check for consistency
unique_winners = set(best_performers.values())
print(f"\nCONSISTENCY ANALYSIS:")
if len(unique_winners) == 1:
    champion = list(unique_winners)[0]
    print(f"   OUTSTANDING CONSISTENCY: {champion} performs best across ALL validation methods!")
    print(f"   This indicates robust and reliable performance regardless of data splitting strategy.")
else:
    print(f"   MIXED RESULTS: Different models excel with different validation approaches")
    print(f"   Models showing best performance: {unique_winners}")
    print(f"   This suggests model performance may be sensitive to specific data characteristics.")

print("\nDETAILED PERFORMANCE INSIGHTS")
print("-" * 40)

for model_name in models.keys():
    print(f"\n   {model_name} Analysis:")
    
    # Cross-validation stability
    cv_std = cross_val_results[model_name]['accuracy']['score_std']
    if cv_std < 0.02:
        stability_rating = "Excellent"
    elif cv_std < 0.05:
        stability_rating = "Good"
    else:
        stability_rating = "Variable"
    
    print(f"    Cross-Validation Stability: {stability_rating} (std: {cv_std:.4f})")
    
    # Performance consistency across hold-out splits
    ho_accuracies = [all_holdout_results[split][model_name]['Accuracy'] 
                    for split in split_configurations.keys()]
    ho_range = max(ho_accuracies) - min(ho_accuracies)
    
    if ho_range < 0.03:
        consistency_rating = "Highly Consistent"
    elif ho_range < 0.07:
        consistency_rating = "Moderately Consistent"
    else:
        consistency_rating = "Inconsistent"
    
    print(f"     Hold-Out Consistency: {consistency_rating} (range: {ho_range:.4f})")
    
    avg_accuracy = np.mean([cross_val_results[model_name]['accuracy']['average_score']] + ho_accuracies)
    if avg_accuracy > 0.98:
        impact = "Excellent - Suitable for production deployment"
    elif avg_accuracy > 0.95:
        impact = "Good - Could be deployed with monitoring"
    elif avg_accuracy > 0.90:
        impact = "Acceptable - May need improvement for critical applications"
    else:
        impact = "Needs significant improvement"
    
    print(f"     Business Readiness: {impact}")

print("\nPERFORMANCE ACROSS ALL VALIDATION METHODS")
print("-" * 45)

# Final summary statistics
print("\nFINAL SUMMARY STATISTICS")
print("-" * 30)

for model_name in models.keys():
    cv_acc = cross_val_results[model_name]['accuracy']['average_score']
    ho_80_acc = all_holdout_results['0.8 train'][model_name]['Accuracy']
    ho_70_acc = all_holdout_results['0.7 train'][model_name]['Accuracy']
    ho_60_acc = all_holdout_results['0.6 train'][model_name]['Accuracy']
    
    avg_performance = np.mean([cv_acc, ho_80_acc, ho_70_acc, ho_60_acc])
    
    print(f"\n{model_name}:")
    print(f"  Average Accuracy Across All Methods: {avg_performance:.3f}")
    print(f"  Best Single Performance: {max([cv_acc, ho_80_acc, ho_70_acc, ho_60_acc]):.3f}")
    print(f"  Performance Range: {max([cv_acc, ho_80_acc, ho_70_acc, ho_60_acc]) - min([cv_acc, ho_80_acc, ho_70_acc, ho_60_acc]):.3f}")